# Production Battery Prognostics & Inference Pipeline

## NASA Li-ion Battery Health Intelligence

This notebook converts the validated battery analytics research workflow into a reproducible, deployment-oriented prognostics pipeline.

The objective is not to introduce additional model complexity. Instead, the focus is on model finalization, reproducibility, serialization, inference safety, and validation of the complete prediction workflow.

## Objectives

1. Consolidate causal battery degradation feature engineering into reusable production code.
2. Train final multi-battery SOH and RUL models using the modelling choices selected through prior validation.
3. Serialize fitted models together with reproducibility metadata.
4. Develop a production-style inference interface for unseen battery telemetry.
5. Apply physical and data-quality guardrails before returning predictions.
6. Verify that serialized models reproduce the original predictions exactly after reload.
7. Test inference behaviour under valid and invalid input conditions.
8. Separate raw model outputs from physically constrained reported outputs.
9. Document model assumptions, intended use, and deployment limitations.
10. Prepare the modelling layer for integration with a Streamlit Battery Intelligence Dashboard.

## Final model-selection rationale

Previous leave-one-battery-out validation across NASA B0005, B0006, B0007, and B0018 showed that:

- Linear models provided the strongest overall cross-battery RUL benchmark among the tested Linear, Ridge, and Random Forest approaches.
- SOH estimation transferred substantially better across batteries than future RUL prediction.
- RUL prediction remained sensitive to battery-specific degradation regimes and calibration shift.
- Random Forest models showed poor target extrapolation behaviour across several held-out batteries.
- Simple feature-space distance metrics did not provide a reliable confidence score.
- SHAP analysis showed that recent historical SOH trajectory contained the dominant predictive signal, while temperature and voltage contributions were more battery-dependent.
- A nonnegative RUL constraint is physically justified because remaining useful life cannot be negative.
- A monotonic running-minimum correction is retained only as an experimental diagnostic because its benefit was not consistent across held-out batteries.

## Production principles

- Predictions must use only information available at or before the prediction time.
- Current-cycle SOH, current capacity, and discharge-cycle count are not used as direct RUL predictors.
- Battery identity is never used as a predictive shortcut.
- Training and inference must use an identical ordered feature schema.
- Models must be serialized together with their preprocessing pipeline.
- Model artifacts must include metadata describing training batteries, EOL threshold, feature definitions, and model version.
- Invalid or insufficient telemetry should raise an explicit validation error rather than silently producing a prediction.
- Raw RUL predictions are retained for diagnostics.
- Reported RUL predictions are constrained to be nonnegative.
- Model outputs are decision-support estimates, not certified battery safety limits.

In [4]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "raw"
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from nasa_loader import load_nasa_battery
from battery_features import add_soh_features
from prognostics import (
    PROGNOSTIC_FEATURES,
    add_causal_prognostic_features,
    identify_eol,
    build_rul_dataset,
    build_soh_dataset,
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Production features: {PROGNOSTIC_FEATURES}")

Project root: C:\Users\Drago\Battery-Analytics-Portfolio
Data directory: C:\Users\Drago\Battery-Analytics-Portfolio\data\raw
Production features: ['SOH_lag1', 'SOH_roll_mean_5', 'SOH_roll_std_5', 'SOH_delta_5', 'temp_roll_mean_5', 'temperature_delta_5', 'voltage_roll_mean_5']


In [5]:
battery_files = {
    "B0005": DATA_DIR / "B0005.mat",
    "B0006": DATA_DIR / "B0006.mat",
    "B0007": DATA_DIR / "B0007.mat",
    "B0018": DATA_DIR / "B0018.mat",
}

batteries = {}

for battery_id, file_path in battery_files.items():
    df = load_nasa_battery(file_path, battery_id)
    df = add_soh_features(df)

    batteries[battery_id] = df

    eol = identify_eol(
        df,
        battery_id=battery_id,
    )

    rul_dataset, _ = build_rul_dataset(
        df,
        battery_id=battery_id,
    )

    soh_dataset = build_soh_dataset(df)

    print(
        f"{battery_id} | "
        f"discharge cycles = {len(df):3d} | "
        f"EOL cycle = {eol.eol_cycle:3d} | "
        f"SOH at EOL = {eol.soh_at_eol:6.2f}% | "
        f"RUL rows = {len(rul_dataset):3d} | "
        f"SOH rows = {len(soh_dataset):3d}"
    )

B0005 | discharge cycles = 168 | EOL cycle = 101 | SOH at EOL =  79.74% | RUL rows =  94 | SOH rows = 162
B0006 | discharge cycles = 168 | EOL cycle =  61 | SOH at EOL =  79.05% | RUL rows =  54 | SOH rows = 162
B0007 | discharge cycles = 168 | EOL cycle = 124 | SOH at EOL =  79.73% | RUL rows = 117 | SOH rows = 162
B0018 | discharge cycles = 132 | EOL cycle =  75 | SOH at EOL =  79.96% | RUL rows =  68 | SOH rows = 126


In [6]:
rul_frames = []
soh_frames = []

for battery_id, df in batteries.items():
    rul_df, eol = build_rul_dataset(
        df,
        battery_id=battery_id,
    )

    soh_df = build_soh_dataset(df)

    rul_df["battery_id"] = battery_id
    soh_df["battery_id"] = battery_id

    rul_frames.append(rul_df)
    soh_frames.append(soh_df)

rul_training_df = pd.concat(
    rul_frames,
    ignore_index=True,
)

soh_training_df = pd.concat(
    soh_frames,
    ignore_index=True,
)

print("Final RUL training dataset")
print(f"Rows: {len(rul_training_df)}")
print(
    rul_training_df.groupby("battery_id")
    .size()
    .to_string()
)

print("\nFinal SOH training dataset")
print(f"Rows: {len(soh_training_df)}")
print(
    soh_training_df.groupby("battery_id")
    .size()
    .to_string()
)

print("\nRUL target range:")
print(
    f"{rul_training_df['RUL_cycles'].min():.0f} "
    f"to {rul_training_df['RUL_cycles'].max():.0f} cycles"
)

print("\nSOH target range:")
print(
    f"{soh_training_df['SOH_percent'].min():.2f}% "
    f"to {soh_training_df['SOH_percent'].max():.2f}%"
)


Final RUL training dataset
Rows: 333
battery_id
B0005     94
B0006     54
B0007    117
B0018     68

Final SOH training dataset
Rows: 612
battery_id
B0005    162
B0006    162
B0007    162
B0018    126

RUL target range:
1 to 117 cycles

SOH target range:
56.69% to 99.75%


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

X_rul = rul_training_df[PROGNOSTIC_FEATURES]
y_rul = rul_training_df["RUL_cycles"]

X_soh = soh_training_df[PROGNOSTIC_FEATURES]
y_soh = soh_training_df["SOH_percent"]


rul_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("regressor", LinearRegression()),
    ]
)

soh_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("regressor", LinearRegression()),
    ]
)


rul_model.fit(X_rul, y_rul)
soh_model.fit(X_soh, y_soh)

print("Final models fitted successfully.")
print(f"RUL training observations: {len(X_rul)}")
print(f"SOH training observations: {len(X_soh)}")
print(f"Number of predictors: {len(PROGNOSTIC_FEATURES)}")

Final models fitted successfully.
RUL training observations: 333
SOH training observations: 612
Number of predictors: 7


In [8]:
import numpy as np
import pandas as pd


def get_model_coefficients(model, feature_names):
    regressor = model.named_steps["regressor"]

    return pd.DataFrame(
        {
            "feature": feature_names,
            "standardized_coefficient": regressor.coef_,
            "abs_coefficient": np.abs(regressor.coef_),
        }
    ).sort_values(
        "abs_coefficient",
        ascending=False,
    )


rul_coefficients = get_model_coefficients(
    rul_model,
    PROGNOSTIC_FEATURES,
)

soh_coefficients = get_model_coefficients(
    soh_model,
    PROGNOSTIC_FEATURES,
)


print("FINAL RUL MODEL")
print(f"Intercept: {rul_model.named_steps['regressor'].intercept_:.4f}")
print(
    rul_coefficients[
        ["feature", "standardized_coefficient"]
    ].to_string(index=False)
)


print("\nFINAL SOH MODEL")
print(f"Intercept: {soh_model.named_steps['regressor'].intercept_:.4f}")
print(
    soh_coefficients[
        ["feature", "standardized_coefficient"]
    ].to_string(index=False)
)


# Numerical sanity checks
rul_fit_predictions = rul_model.predict(X_rul)
soh_fit_predictions = soh_model.predict(X_soh)

print("\nPrediction sanity checks")

print(
    f"RUL prediction range: "
    f"{rul_fit_predictions.min():.2f} to "
    f"{rul_fit_predictions.max():.2f} cycles"
)

print(
    f"Negative raw RUL predictions: "
    f"{np.sum(rul_fit_predictions < 0)}"
)

print(
    f"SOH prediction range: "
    f"{soh_fit_predictions.min():.2f}% to "
    f"{soh_fit_predictions.max():.2f}%"
)

assert np.isfinite(rul_fit_predictions).all()
assert np.isfinite(soh_fit_predictions).all()

print("\nAll fitted predictions are finite.")

FINAL RUL MODEL
Intercept: 45.6426
            feature  standardized_coefficient
    SOH_roll_mean_5                 24.007507
           SOH_lag1                 13.522936
voltage_roll_mean_5                 -9.529885
   temp_roll_mean_5                  8.401505
     SOH_roll_std_5                 -4.207220
temperature_delta_5                 -0.124020
        SOH_delta_5                  0.002021

FINAL SOH MODEL
Intercept: 82.2062
            feature  standardized_coefficient
           SOH_lag1                  8.555011
    SOH_roll_mean_5                  2.003810
     SOH_roll_std_5                 -0.208715
   temp_roll_mean_5                 -0.071782
temperature_delta_5                 -0.052950
        SOH_delta_5                  0.028994
voltage_roll_mean_5                  0.016176

Prediction sanity checks
RUL prediction range: -6.65 to 113.82 cycles
Negative raw RUL predictions: 7
SOH prediction range: 56.75% to 99.41%

All fitted predictions are finite.


In [9]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)


def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "Bias": np.mean(y_pred - y_true),
    }


# Raw fitted predictions
rul_raw_fit = rul_model.predict(X_rul)
soh_fit = soh_model.predict(X_soh)

# Physically constrained RUL
rul_reported_fit = np.maximum(rul_raw_fit, 0.0)


rul_raw_metrics = regression_metrics(
    y_rul,
    rul_raw_fit,
)

rul_reported_metrics = regression_metrics(
    y_rul,
    rul_reported_fit,
)

soh_fit_metrics = regression_metrics(
    y_soh,
    soh_fit,
)


metrics_table = pd.DataFrame(
    [
        {
            "Model": "RUL - raw",
            **rul_raw_metrics,
        },
        {
            "Model": "RUL - nonnegative",
            **rul_reported_metrics,
        },
        {
            "Model": "SOH",
            **soh_fit_metrics,
        },
    ]
)


print("POST-SELECTION FULL-DATA FIT DIAGNOSTICS")
print(
    metrics_table.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)

print("\nImportant:")
print(
    "These are in-sample fit diagnostics for the final "
    "all-battery artifacts."
)
print(
    "They are NOT estimates of unseen-battery "
    "generalization performance."
)
print(
    "Cross-battery generalization evidence comes from "
    "the LOBO validation performed in Notebook 04."
)

POST-SELECTION FULL-DATA FIT DIAGNOSTICS
            Model    MAE   RMSE     R2   Bias
        RUL - raw 6.9501 8.5408 0.9161 0.0000
RUL - nonnegative 6.8753 8.4870 0.9171 0.0748
              SOH 0.4610 0.9034 0.9928 0.0000

Important:
These are in-sample fit diagnostics for the final all-battery artifacts.
They are NOT estimates of unseen-battery generalization performance.
Cross-battery generalization evidence comes from the LOBO validation performed in Notebook 04.


In [10]:
import json
import platform
from datetime import datetime, timezone

import joblib
import sklearn


MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

SOH_MODEL_PATH = MODELS_DIR / "soh_model.joblib"
RUL_MODEL_PATH = MODELS_DIR / "rul_model.joblib"
METADATA_PATH = MODELS_DIR / "model_metadata.json"


# Save complete sklearn pipelines:
# StandardScaler -> LinearRegression
joblib.dump(
    soh_model,
    SOH_MODEL_PATH,
)

joblib.dump(
    rul_model,
    RUL_MODEL_PATH,
)


metadata = {
    "model_version": "1.0.0",
    "created_utc": datetime.now(timezone.utc).isoformat(),

    "project": "Battery Analytics Portfolio",

    "model_selection": {
        "soh_model": "StandardScaler + LinearRegression",
        "rul_model": "StandardScaler + LinearRegression",
        "selection_basis":
            "Leave-one-battery-out validation in Notebook 04",
    },

    "training": {
        "battery_ids": [
            "B0005",
            "B0006",
            "B0007",
            "B0018",
        ],
        "rul_training_rows": int(len(rul_training_df)),
        "soh_training_rows": int(len(soh_training_df)),
    },

    "feature_engineering": {
        "causal": True,
        "history_window_cycles": 5,
        "feature_order": PROGNOSTIC_FEATURES,
        "first_usable_discharge_cycle": 7,
    },

    "targets": {
        "soh": {
            "name": "SOH_percent",
            "unit": "percent",
        },
        "rul": {
            "name": "RUL_cycles",
            "unit": "discharge_cycles",
            "eol_threshold_soh_percent": 80.0,
        },
    },

    "inference_policy": {
        "rul_raw_output_retained": True,
        "rul_reporting_constraint": "max(raw_rul, 0.0)",
        "monotonic_running_minimum": False,
    },

    "final_fit_diagnostics": {
        "rul_raw": {
            key: float(value)
            for key, value in rul_raw_metrics.items()
        },
        "rul_nonnegative": {
            key: float(value)
            for key, value in rul_reported_metrics.items()
        },
        "soh": {
            key: float(value)
            for key, value in soh_fit_metrics.items()
        },
    },

    "validation_note": (
        "Final-fit diagnostics are in-sample and must not be "
        "interpreted as unseen-battery performance. "
        "Cross-battery generalization evidence is provided by "
        "leave-one-battery-out validation in Notebook 04."
    ),

    "limitations": [
        "Training data contains four NASA laboratory-aged Li-ion cells.",
        "RUL generalization is battery-regime dependent.",
        "The 80 percent SOH EOL threshold is a modelling convention.",
        "No validated probabilistic uncertainty interval is provided.",
        "The model is not a certified battery safety or BMS control system.",
    ],

    "software": {
        "python": platform.python_version(),
        "scikit_learn": sklearn.__version__,
        "joblib": joblib.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
}


with open(
    METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
    )


print("Production artifacts saved:")
print(f"  SOH model : {SOH_MODEL_PATH}")
print(f"  RUL model : {RUL_MODEL_PATH}")
print(f"  Metadata  : {METADATA_PATH}")

print("\nArtifact sizes:")
print(
    f"  SOH model : "
    f"{SOH_MODEL_PATH.stat().st_size / 1024:.2f} KB"
)
print(
    f"  RUL model : "
    f"{RUL_MODEL_PATH.stat().st_size / 1024:.2f} KB"
)
print(
    f"  Metadata  : "
    f"{METADATA_PATH.stat().st_size / 1024:.2f} KB"
)

Production artifacts saved:
  SOH model : C:\Users\Drago\Battery-Analytics-Portfolio\models\soh_model.joblib
  RUL model : C:\Users\Drago\Battery-Analytics-Portfolio\models\rul_model.joblib
  Metadata  : C:\Users\Drago\Battery-Analytics-Portfolio\models\model_metadata.json

Artifact sizes:
  SOH model : 1.65 KB
  RUL model : 1.65 KB
  Metadata  : 2.45 KB


In [11]:
# Reload serialized artifacts from disk
loaded_soh_model = joblib.load(SOH_MODEL_PATH)
loaded_rul_model = joblib.load(RUL_MODEL_PATH)

with open(
    METADATA_PATH,
    "r",
    encoding="utf-8",
) as file:
    loaded_metadata = json.load(file)


# Generate predictions from reloaded models
reloaded_rul_predictions = loaded_rul_model.predict(X_rul)
reloaded_soh_predictions = loaded_soh_model.predict(X_soh)


# Compare against the original in-memory models
rul_max_abs_difference = np.max(
    np.abs(
        reloaded_rul_predictions
        - rul_raw_fit
    )
)

soh_max_abs_difference = np.max(
    np.abs(
        reloaded_soh_predictions
        - soh_fit
    )
)


print("SERIALIZATION PARITY CHECK")

print(
    f"RUL max absolute prediction difference: "
    f"{rul_max_abs_difference:.12e}"
)

print(
    f"SOH max absolute prediction difference: "
    f"{soh_max_abs_difference:.12e}"
)


assert np.allclose(
    reloaded_rul_predictions,
    rul_raw_fit,
    rtol=1e-12,
    atol=1e-12,
)

assert np.allclose(
    reloaded_soh_predictions,
    soh_fit,
    rtol=1e-12,
    atol=1e-12,
)

assert (
    loaded_metadata["feature_engineering"]["feature_order"]
    == PROGNOSTIC_FEATURES
)

assert (
    loaded_metadata["targets"]["rul"][
        "eol_threshold_soh_percent"
    ]
    == 80.0
)


print("\nModel reload parity: PASSED")
print("Feature-schema metadata check: PASSED")
print("EOL-threshold metadata check: PASSED")

SERIALIZATION PARITY CHECK
RUL max absolute prediction difference: 0.000000000000e+00
SOH max absolute prediction difference: 0.000000000000e+00

Model reload parity: PASSED
Feature-schema metadata check: PASSED
EOL-threshold metadata check: PASSED


In [12]:
from inference import BatteryPrognosticsEngine


engine = BatteryPrognosticsEngine(
    MODELS_DIR
)

print("Inference engine loaded successfully.")
print(f"Model version: {engine.model_version}")
print(
    f"EOL threshold: "
    f"{engine.eol_threshold:.1f}% SOH"
)

Inference engine loaded successfully.
Model version: 1.0.0
EOL threshold: 80.0% SOH


In [13]:
# End-to-end inference test on B0005 before EOL

test_cycle = 90

b0005_history = batteries["B0005"].loc[
    batteries["B0005"]["cycle"] <= test_cycle
].copy()

result = engine.predict(
    b0005_history
)

actual_rul = 101 - test_cycle

print("END-TO-END INFERENCE TEST — B0005")

for key, value in result.items():
    print(f"{key}: {value}")

print(
    f"\nObserved historical EOL cycle: 101"
)

print(
    f"Actual RUL at cycle {test_cycle}: "
    f"{actual_rul} cycles"
)

print(
    f"Reported prediction error: "
    f"{result['reported_rul_cycles'] - actual_rul:.2f} cycles"
)

END-TO-END INFERENCE TEST — B0005
model_version: 1.0.0
cycle: 90
observed_soh_percent: 86.49769888680268
predicted_soh_percent: 81.64938733078588
raw_rul_cycles: 9.04890445059894
reported_rul_cycles: 9.04890445059894
eol_threshold_soh_percent: 80.0
eol_reached: False

Observed historical EOL cycle: 101
Actual RUL at cycle 90: 11 cycles
Reported prediction error: -1.95 cycles


In [14]:
# EOL safeguard test

test_cycle_eol = 101

b0005_eol_history = batteries["B0005"].loc[
    batteries["B0005"]["cycle"] <= test_cycle_eol
].copy()

eol_result = engine.predict(
    b0005_eol_history
)

print("EOL SAFEGUARD TEST — B0005")

for key, value in eol_result.items():
    print(f"{key}: {value}")


assert eol_result["eol_reached"] is True
assert eol_result["reported_rul_cycles"] == 0.0

print("\nEOL safeguard: PASSED")

print(
    "Raw RUL is retained for diagnostics, "
    "but reported RUL is forced to zero "
    "after the observed EOL threshold is reached."
)

EOL SAFEGUARD TEST — B0005
model_version: 1.0.0
cycle: 101
observed_soh_percent: 79.74272604140161
predicted_soh_percent: 79.91656441560693
raw_rul_cycles: 1.4589710931709376
reported_rul_cycles: 0.0
eol_threshold_soh_percent: 80.0
eol_reached: True

EOL safeguard: PASSED
Raw RUL is retained for diagnostics, but reported RUL is forced to zero after the observed EOL threshold is reached.


In [15]:
def expect_failure(
    name,
    history,
    expected_exception=ValueError,
):
    try:
        engine.predict(history)

    except expected_exception as exc:
        print(f"{name}: PASSED")
        print(
            f"  {type(exc).__name__}: {exc}"
        )

    else:
        raise AssertionError(
            f"{name}: expected "
            f"{expected_exception.__name__}"
        )


print("INFERENCE GUARDRAIL TESTS\n")


# 1. Insufficient degradation history
short_history = batteries["B0005"].iloc[:5].copy()

expect_failure(
    "Insufficient history",
    short_history,
)


# 2. Duplicate discharge cycle
duplicate_cycle_history = (
    batteries["B0005"]
    .iloc[:20]
    .copy()
)

duplicate_cycle_history.loc[
    duplicate_cycle_history.index[-1],
    "cycle",
] = duplicate_cycle_history.iloc[-2]["cycle"]

expect_failure(
    "Duplicate cycle",
    duplicate_cycle_history,
)


# 3. Missing required telemetry column
missing_column_history = (
    batteries["B0005"]
    .iloc[:20]
    .drop(columns=["max_temperature_C"])
    .copy()
)

expect_failure(
    "Missing telemetry column",
    missing_column_history,
)


# 4. Missing numerical value
missing_value_history = (
    batteries["B0005"]
    .iloc[:20]
    .copy()
)

missing_value_history.loc[
    missing_value_history.index[5],
    "SOH_percent",
] = np.nan

expect_failure(
    "Missing numerical value",
    missing_value_history,
)


# 5. Non-chronological history
unordered_history = (
    batteries["B0005"]
    .iloc[:20]
    .copy()
)

unordered_history = pd.concat(
    [
        unordered_history.iloc[:10],
        unordered_history.iloc[10:].iloc[::-1],
    ]
)

expect_failure(
    "Non-chronological history",
    unordered_history,
)


print(
    "\nAll inference guardrail tests completed."
)

INFERENCE GUARDRAIL TESTS

Insufficient history: PASSED
  ValueError: Insufficient battery history to construct production prognostic features. Unavailable features: ['SOH_roll_mean_5', 'SOH_roll_std_5', 'SOH_delta_5', 'temp_roll_mean_5', 'temperature_delta_5', 'voltage_roll_mean_5']
Duplicate cycle: PASSED
  ValueError: Duplicate discharge-cycle values detected.
Missing telemetry column: PASSED
  ValueError: Missing required battery columns: ['max_temperature_C']
Missing numerical value: PASSED
  ValueError: Column 'SOH_percent' contains missing values.
Non-chronological history: PASSED
  ValueError: Battery history must be ordered by strictly increasing discharge cycle.

All inference guardrail tests completed.
